In [1]:
# Cell 1: Imports
import torch
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
import time
from typing import List, Tuple
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset


Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1221 21:48:06.465000 16096 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# Cell 2: Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print()

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
Memory allocated: 0.00 GB
Loading model: google/flan-t5-base...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Model loaded: google/flan-t5-base
Model parameters: 247,577,856
Tokenizer vocab size: 32,000



In [3]:
# Cell 3: Define dataset class
class ConversationDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=128):
        self.data = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prefix = "Write a reply in your normal texting style:"
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        message = str(self.data.iloc[idx]['message'])
        response = str(self.data.iloc[idx]['response'])
        
        input_text = self.prefix + message
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            response,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'labels': labels.squeeze(0)
        }

In [4]:
# Cell 4: Load Your Data
train_csv_path = "../data/processed/mr_train.csv"
val_csv_path = "../data/processed/mr_val.csv"

train_dataset = ConversationDataset(train_csv_path, tokenizer)
val_dataset = ConversationDataset(val_csv_path, tokenizer)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

Train samples: 9388
Val samples: 2348


In [5]:
# Cell 5: Create DataLoaders
batch_size = 8

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")

Train batches: 1174
Val batches: 294


In [6]:
# Cell 6: Training Function
def train_simple(model, train_loader, val_loader, epochs=3, lr=3e-4):
    # Use Adam optimizer (not AdamW)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        
        # Training
        model.train()
        train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            train_loss += loss.item()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Print progress every 10 batches
            if batch_idx % 10 == 0:
                print(f"  Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                val_loss += outputs.loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"  Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    return model

In [7]:
# Cell 7: Train Model
print("Starting training...")
trained_model = train_simple(
    model,
    train_dataloader,
    val_dataloader,
    epochs=3,
    lr=3e-4
)
print("Training complete!")

Starting training...
Epoch 1/3
  Batch 0/1174, Loss: 3.8896
  Batch 10/1174, Loss: 4.9006
  Batch 20/1174, Loss: 5.4528
  Batch 30/1174, Loss: 4.8112
  Batch 40/1174, Loss: 4.3063
  Batch 50/1174, Loss: 4.2696
  Batch 60/1174, Loss: 4.0103
  Batch 70/1174, Loss: 5.0208
  Batch 80/1174, Loss: 4.1165
  Batch 90/1174, Loss: 4.2636
  Batch 100/1174, Loss: 2.0973
  Batch 110/1174, Loss: 3.4573
  Batch 120/1174, Loss: 3.7537
  Batch 130/1174, Loss: 4.3931
  Batch 140/1174, Loss: 4.2024
  Batch 150/1174, Loss: 4.2335
  Batch 160/1174, Loss: 3.7812
  Batch 170/1174, Loss: 4.5519
  Batch 180/1174, Loss: 3.1015
  Batch 190/1174, Loss: 3.6344
  Batch 200/1174, Loss: 4.4036
  Batch 210/1174, Loss: 4.6946
  Batch 220/1174, Loss: 4.6574
  Batch 230/1174, Loss: 3.2089
  Batch 240/1174, Loss: 4.2285
  Batch 250/1174, Loss: 3.8280
  Batch 260/1174, Loss: 4.7156
  Batch 270/1174, Loss: 3.7987
  Batch 280/1174, Loss: 2.8799
  Batch 290/1174, Loss: 3.8073
  Batch 300/1174, Loss: 3.7020
  Batch 310/1174, L

In [8]:
# Cell 8: Generate Multiple Responses Function
def generate_responses(model, tokenizer, message, device, num_responses=3):
    input_text = "Reply: " + message
    input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_length=64,
            num_beams=5,
            num_return_sequences=num_responses,
            early_stopping=True
        )
        # output_ids = model.generate(
        #     input_ids=input_ids,
        #     max_new_tokens=50,
        #     num_return_sequences=num_responses,
        #     do_sample=True,
        #     temperature=0.3,  # Increased from 0.8 (more randomness)
        #     top_p=0.9,       # Nucleus sampling for diversity
        #     top_k=50,        # Limit to top 50 tokens
        #     repetition_penalty=1.2,  # Lower penalty = more generic
        #     num_beams=5,     # Add beams for better quality
        # )
        
		
    
    responses = []
    for i in range(num_responses):
        response = tokenizer.decode(output_ids[i], skip_special_tokens=True)
        responses.append(response)
    
    return responses

In [9]:
# Cell 9: Test Model
test_messages = [
    "Hey, what's up?",
    "How are you doing?",
    "See you tomorrow",
    "have you finished the report?"
]

print("Testing model...")
for msg in test_messages:
    response = generate_responses(trained_model, tokenizer, msg, device)
    print(f"Input: {msg}")
    print(f"Response: {response}")
    print("-" * 40)

Testing model...
Input: Hey, what's up?
Response: ['what the hael', 'no idea what that is', "what's up bro"]
----------------------------------------
Input: How are you doing?
Response: ["i'm doing math right now", "i don't know", "i don't know yet"]
----------------------------------------
Input: See you tomorrow
Response: ["i'll see you then", "i'll be home by then", "yeah i'll see you then"]
----------------------------------------
Input: have you finished the report?
Response: ["no i haven't finished", "i haven't finished yet", 'no i have not finished']
----------------------------------------


In [10]:
# Cell 10: Save Model
save_path = "../models/fine_tuned"
trained_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Model saved to ../models/fine_tuned
